# Phase 0 inspection: does this idea work at all?

This notebook is the human half of the feasibility gate. The scripts in `scripts/phase0.py` pulled a few hundred Mapillary frames from one corridor, ran MegaDetector + SpeciesNet over them, and wrote everything to DuckDB. Before anything gets built on top of that, I want to look at what the model actually flagged and answer the five questions from the brief honestly: how often does it fire, how often is it right, how far out does it work, does it know what it is looking at, and how much imagery is there to begin with.

Nothing here modifies the raw model output. My verdicts go into a separate `manual_review` table by way of the CSV in `data/review/<corridor>/`, so the accuracy numbers can always be recomputed from the untouched predictions.

The notebook expects the light environment (`make setup`) and the `parkwild (.venv)` kernel. It does not need PyTorch.

In [ ]:
# One writable connection for the whole notebook. DuckDB refuses to open the same
# file twice in one process with different settings, so I don't mix read-only and
# writable connections here. The analysis cells only read; the single cell that
# writes is the verdict import further down.
from pathlib import Path

import pandas as pd
from IPython.display import HTML, display
from PIL import Image

from parkwild.config import DB_PATH, REVIEW_DIR
from parkwild.storage import Store

CORRIDOR = "lamar_valley"   # change to whichever corridor `phase0.py coverage` picked

store = Store(DB_PATH)
pd.set_option("display.max_colwidth", 90)
print("connected to", DB_PATH)

## How much came through

The first number is the crawl index for the whole corridor bbox. The next two are the subsample that was actually downloaded and scored. If `scored` is much smaller than `downloaded`, SpeciesNet failed on some files and `phase0.py detect` should be re-run (it resumes, so only the missing ones get processed).

In [ ]:
volume = store.df(
    """
    SELECT
      (SELECT count(*) FROM images WHERE corridor = ?) AS indexed,
      (SELECT count(*) FROM downloads d JOIN images i USING (image_id)
        WHERE i.corridor = ? AND d.error IS NULL) AS downloaded,
      (SELECT count(*) FROM predictions_raw p JOIN images i USING (image_id)
        WHERE i.corridor = ?) AS scored,
      (SELECT count(*) FROM predictions_raw p JOIN images i USING (image_id)
        WHERE i.corridor = ? AND p.failures IS NOT NULL) AS model_failures
    """,
    [CORRIDOR] * 4,
)
volume

## Question 1: how often does the detector fire?

The brief asks for the fraction of images with any animal box at 0.2 confidence. I sweep a few thresholds as well, because the number at 0.2 alone does not say whether those are confident hits or a long tail of noise. `max_animal_conf` is the strongest animal box in each image, so this is a per-image count, not a per-box count. Human and vehicle boxes are shown for context: a detector that finds every car but no animals is working fine, there just are no animals.

In [ ]:
sweep = store.df(
    """
    SELECT t AS threshold,
           count(*) FILTER (WHERE p.max_animal_conf >= t) AS images_with_animal,
           round(100.0 * count(*) FILTER (WHERE p.max_animal_conf >= t) / count(*), 1) AS pct
    FROM predictions_raw p
    JOIN images i USING (image_id),
         (SELECT unnest([0.1, 0.2, 0.3, 0.5, 0.7, 0.8, 0.9]) AS t)
    WHERE i.corridor = ?
    GROUP BY t ORDER BY t
    """,
    [CORRIDOR],
)
display(sweep)

context = store.df(
    """
    SELECT d.label, count(DISTINCT d.image_id) AS images
    FROM detections_raw d JOIN images i USING (image_id)
    WHERE i.corridor = ? AND d.conf >= 0.2
    GROUP BY d.label ORDER BY images DESC
    """,
    [CORRIDOR],
)
display(context)

### What the ensemble thinks those are

SpeciesNet's final label per image, restricted to frames with an animal box at or above 0.2. Labels like "deer family" are deliberate: when the classifier is unsure between elk and mule deer it rolls up to the lowest taxon it is confident about. A wall of "blank" or "unknown" next to a high detector score is the domain-shift signature the brief warns about.

In [ ]:
from parkwild.speciesnet_runner import display_name

labels = store.df(
    """
    SELECT p.prediction, p.prediction_source, count(*) AS images,
           round(avg(p.prediction_score), 2) AS mean_score
    FROM predictions_raw p JOIN images i USING (image_id)
    WHERE i.corridor = ? AND p.max_animal_conf >= 0.2
    GROUP BY 1, 2 ORDER BY images DESC
    """,
    [CORRIDOR],
)
labels["prediction"] = labels["prediction"].map(display_name)
labels

## Questions 2 to 4: look at the boxes

`phase0.py sample` picked ~30 animal boxes at random, one per frame, and rendered two images for each: the whole frame with every animal box drawn (the sampled one in orange, others in yellow, confidence printed above each), and a padded crop of the sampled box, upscaled with nearest-neighbour so I am judging the real pixels and not a smoothed blur.

I go through them below and fill in `review.csv`:

| column | what I write |
|---|---|
| `verdict` | `tp` (an animal is in the box), `fp` (rock, shrub, log, shadow, car part), `unsure` |
| `true_species` | what I think it actually is, as specific as I can be |
| `species_agree` | `yes` if the model's label matches, `rollup` if the model gave a correct coarser taxon (e.g. "deer family" for an elk), `no` if wrong, `na` if the model said blank/unknown |
| `est_distance_m` | rough range from camera to animal, rounded to 25 m |
| `notes` | anything useful: herd, partially occluded, in the road, etc. |

Distance is a judgement call. My rulers: a two-lane park road is about 7 m wide, a pickup is about 5.5 m long, an adult bison is about 3 m long and 1.8 m at the shoulder, an elk about 2.4 m long. Anything past ~400 m gets `400+`-style honesty in the notes and a `400` in the number.

In [ ]:
review_dir = REVIEW_DIR / CORRIDOR
review_csv = review_dir / "review.csv"
review = pd.read_csv(review_csv, dtype=str).fillna("")
print(f"{len(review)} boxes in the review set; {(review.verdict != '').sum()} judged so far")

def show(row, frame_width=900, crop_width=420):
    frame = Image.open(review_dir / row.frame_file)
    frame.thumbnail((frame_width, frame_width))
    crop = Image.open(review_dir / row.crop_file)
    crop.thumbnail((crop_width, crop_width))
    display(HTML(
        f"<hr><b>{row.image_id}</b> &middot; box {row.det_idx} &middot; detector {row.conf} &middot; "
        f"model says <b>{row.predicted}</b> ({row.prediction_score}, {row.prediction_source}) &middot; "
        f"<a href='{row.source_url}' target='_blank'>open on Mapillary</a>"
        f"<br><small>top-5: {row.top5}</small>"
        f"<br><small>verdict so far: <b>{row.verdict or '-'}</b> {row.true_species} {row.est_distance_m}</small>"
    ))
    display(frame)
    display(crop)

# Page through the set a few at a time; 30 full frames inline is a lot of scrolling.
START, N = 0, 10
for _, row in review.iloc[START:START + N].iterrows():
    show(row)

## Load the verdicts

Once `review.csv` has verdicts, this cell writes them into `manual_review`. Re-running is safe: rows are upserted by `(image_id, det_idx, reviewer)`, and the raw prediction tables are never touched. `phase0.py report` does the same import, so either path works.

In [ ]:
from parkwild.review import load_review_csv

verdicts = load_review_csv(review_csv, reviewer="me")
store.upsert_reviews(verdicts)
print(f"loaded {len(verdicts)} verdicts into manual_review")

## Precision, distance, species agreement

The numbers the brief asks for, computed from the review table joined back to the raw detections. Precision by confidence band matters as much as the headline: if everything above 0.8 is real and everything between 0.2 and 0.5 is shrubs, the fix is a threshold, not a model.

In [ ]:
judged = store.df(
    """
    SELECT m.verdict, m.species_agree, m.est_distance_m, m.true_species, m.notes,
           d.conf, p.prediction, p.prediction_score
    FROM manual_review m
    JOIN detections_raw d USING (image_id, det_idx)
    JOIN predictions_raw p USING (image_id, model_version)
    JOIN images i USING (image_id)
    WHERE i.corridor = ? AND m.reviewer = 'me'
    """,
    [CORRIDOR],
)
judged["prediction"] = judged["prediction"].map(display_name)
tp = judged[judged.verdict == "tp"]
fp = judged[judged.verdict == "fp"]
print(f"reviewed {len(judged)}  |  tp {len(tp)}  |  fp {len(fp)}  |  unsure {(judged.verdict == 'unsure').sum()}")
if len(tp) + len(fp):
    print(f"precision (tp / (tp + fp)): {len(tp) / (len(tp) + len(fp)):.0%}")

print("\nprecision by detector confidence band:")
bands = pd.cut(judged.conf, [0.2, 0.5, 0.8, 1.0], include_lowest=True)
display(judged.assign(band=bands).groupby("band", observed=True).verdict.value_counts().unstack(fill_value=0))

print("\nspecies agreement among true positives:")
display(tp.species_agree.value_counts(dropna=False).rename_axis("species_agree").to_frame("boxes"))

print("\nwhat the true positives actually were vs. what the model said:")
display(tp[["true_species", "prediction", "prediction_score", "est_distance_m", "notes"]].sort_values("est_distance_m"))

### How far out does detection work?

One series, so no legend. The shape is what matters: a wall at some distance tells me the practical range of whole-image detection at this resolution, and whether tiled inference (Phase 2's fallback) is worth the compute.

In [ ]:
import matplotlib.pyplot as plt

dist = tp.est_distance_m.dropna()
if len(dist):
    edges = range(0, int(dist.max()) + 50, 25)
    fig, ax = plt.subplots(figsize=(7.5, 3.2))
    ax.hist(dist, bins=edges, color="#2a78d6", edgecolor="#fcfcfb", linewidth=1.5)
    ax.set_title("Estimated distance of confirmed animals from the camera", loc="left", fontsize=11, color="#0b0b0b")
    ax.set_xlabel("metres (my estimate, rounded to 25 m)", color="#52514e")
    ax.set_ylabel("true positives", color="#52514e")
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.grid(axis="y", color="#e5e5e2", linewidth=0.8)
    ax.set_axisbelow(True)
    plt.tight_layout()
    plt.show()
    print(f"median {dist.median():.0f} m  |  p90 {dist.quantile(0.9):.0f} m  |  farthest confirmed {dist.max():.0f} m")
else:
    print("no distance estimates yet; fill est_distance_m in review.csv")

## Question 5: how much imagery is there, and when?

Density is images per kilometre of road (the road length comes from Overpass in `phase0.py report`; here I just show counts). The by-month table is the seasonal-bias check from the brief: if nearly everything was shot in July, the dataset measures tourism, not animals.

In [ ]:
dates = store.df(
    """
    SELECT min(captured_at) AS first, max(captured_at) AS last, count(*) AS images,
           count(DISTINCT sequence_id) AS sequences, count(DISTINCT creator_username) AS contributors,
           sum(CASE WHEN is_pano THEN 1 ELSE 0 END) AS panoramas
    FROM images WHERE corridor = ?
    """,
    [CORRIDOR],
)
display(dates)

by_year = store.df("SELECT year(captured_at) AS year, count(*) AS images FROM images WHERE corridor = ? GROUP BY 1 ORDER BY 1", [CORRIDOR])
display(by_year.set_index("year").T)

by_month = store.df("SELECT month(captured_at) AS month, count(*) AS images FROM images WHERE corridor = ? GROUP BY 1 ORDER BY 1", [CORRIDOR])
fig, ax = plt.subplots(figsize=(7.5, 3.0))
ax.bar(by_month["month"], by_month["images"], color="#2a78d6", edgecolor="#fcfcfb", linewidth=1.5, width=0.8)
ax.set_title("Images captured, by calendar month (all years)", loc="left", fontsize=11, color="#0b0b0b")
ax.set_xticks(range(1, 13))
ax.set_xticklabels(["J", "F", "M", "A", "M", "J", "J", "A", "S", "O", "N", "D"], color="#52514e")
ax.set_ylabel("images", color="#52514e")
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
ax.grid(axis="y", color="#e5e5e2", linewidth=0.8)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

## Verdict

`phase0.py report --write` puts the numbers into `RESULTS.md`. The conclusion is written by hand here and copied there, because the numbers do not decide by themselves.

The brief's stop rule: if the true-positive rate is under about 2% of frames, or the boxes are mostly vegetation, say so plainly and stop. A negative result is a useful outcome.

**Decision:** _not yet run_

**What I saw:**

- 
- 

**What would change my mind:** 

In [ ]:
store.close()